# 02 — Baseline: Random Forest on Morgan fingerprints

Phase 2 of **CRC_Inhibitor_ML**. Establishes a *number to beat* for the GNN. Workflow:

1. **Load** the clean CSV from Phase 1
2. **Featurize** every SMILES into a 2048-bit Morgan fingerprint (ECFP4)
3. **Compute Bemis-Murcko scaffolds** for scaffold-split evaluation
4. **Train + evaluate** a Random Forest per target, using BOTH random split (easy benchmark) and scaffold split (realistic benchmark)
5. **Save** baseline metrics to `reports/baseline_rf_metrics.json` for direct comparison against the GNN in Phase 3

**Why random + scaffold split:** random split gives the rosy number (test set looks like train set). Scaffold split gives the *real* number (test set has chemistries the model has never seen). The gap between them tells you how much the model is generalizing vs. memorizing — a critical thing to know for real drug-discovery deployment, where you're predicting novel chemistry.

In [ ]:
from pathlib import Path
from collections import defaultdict
import json
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, DataStructs
from rdkit.Chem.Scaffolds import MurckoScaffold

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

RDLogger.DisableLog("rdApp.*")
tqdm.pandas()

PROJECT_ROOT  = Path(r"C:\Users\palla\OneDrive\Documents\Coding Projects\CRC_Inhibitor_ML")
CLEAN_CSV     = PROJECT_ROOT / "data" / "interim"   / "chembl_crc_targets_clean.csv"
REPORTS_DIR   = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
METRICS_PATH  = REPORTS_DIR / "baseline_rf_metrics.json"

TARGETS = {
    "CHEMBL2189121": "KRAS",
    "CHEMBL5145":    "BRAF",
    "CHEMBL203":     "EGFR",
    "CHEMBL4005":    "PIK3CA",
}

print(f"Input:    {CLEAN_CSV}")
print(f"Output:   {METRICS_PATH}")

## Cell 1 — load clean data

Read the curated CSV from Phase 1. Confirm row counts per target are healthy.

In [ ]:
clean = pd.read_csv(CLEAN_CSV)
print(f"Loaded {len(clean):,} rows")
print()
print("Per-target counts:")
print(clean["target_chembl_id"].map(TARGETS).value_counts())
clean.head(3)

## Cell 2 — featurize: Morgan fingerprints + Murcko scaffolds

Two computations per molecule:

### Morgan fingerprint (ECFP4)
A 2048-bit binary vector where each bit = "does this molecule contain a specific local substructure within a 2-bond radius of any atom?" — basically a structural barcode. The model input. Industry standard for cheminformatics ML.

### Bemis-Murcko scaffold
The molecule stripped down to just its **ring system + connecting atoms** (side chains removed). Two molecules with the same scaffold are structurally cousins — like a methylated vs ethylated version of the same core drug. Used for scaffold splitting in Cell 3.

Takes ~30 seconds. Anything that fails to parse gets dropped.

In [ ]:
N_BITS = 2048
RADIUS = 2  # ECFP4

# Parse all SMILES into RDKit Mol objects in one pass; track which ones failed
print("Parsing SMILES...")
mols = [Chem.MolFromSmiles(s) for s in tqdm(clean["canonical_smiles_std"])]
valid_mask = np.array([m is not None for m in mols])

# Filter clean df + mols list to only the parseable ones
clean = clean[valid_mask].reset_index(drop=True)
mols  = [m for m, ok in zip(mols, valid_mask) if ok]
print(f"Parsed: {len(mols):,} valid molecules ({(~valid_mask).sum()} dropped)")

# Morgan fingerprints → numpy matrix (rows = molecules, cols = bits)
print("\nComputing Morgan fingerprints...")
fps = np.zeros((len(mols), N_BITS), dtype=np.int8)
for i, m in enumerate(tqdm(mols)):
    fp = AllChem.GetMorganFingerprintAsBitVect(m, RADIUS, N_BITS)
    DataStructs.ConvertToNumpyArray(fp, fps[i])

# Murcko scaffolds → list of SMILES strings
print("\nComputing Murcko scaffolds...")
scaffolds = []
for m in tqdm(mols):
    try:
        scaffolds.append(MurckoScaffold.MurckoScaffoldSmiles(mol=m, includeChirality=False))
    except Exception:
        scaffolds.append("")  # empty scaffold groups all failures together
clean["scaffold"] = scaffolds

print(f"\nFingerprint matrix: {fps.shape}")
print(f"Unique scaffolds:   {clean['scaffold'].nunique():,}")

## Cell 3 — scaffold split function

Groups molecules by Bemis-Murcko scaffold, then assigns whole scaffold groups to train or test (so no scaffold appears in both splits). This is the realistic evaluation setup for drug discovery: the model never sees molecules with the same chemical framework during training, then has to predict on novel frameworks at test time.

Random split, by contrast, would let a molecule end up in train while its close analog ends up in test — the model can just memorize the analog and look brilliant on the test set. Scaffold split breaks that illusion.

In [ ]:
def scaffold_split_indices(scaffolds, frac_train=0.8):
    """
    Bemis-Murcko scaffold split. All molecules sharing a scaffold go to the
    same split. Largest scaffold groups go to train; smaller (rarer) ones
    fill test. Deterministic given input order.
    """
    scaffold_to_indices = defaultdict(list)
    for i, s in enumerate(scaffolds):
        scaffold_to_indices[s].append(i)

    # Sort scaffold groups by size, largest first
    groups = sorted(scaffold_to_indices.values(), key=len, reverse=True)

    train_cutoff = int(frac_train * len(scaffolds))
    train_idx, test_idx = [], []
    for indices in groups:
        if len(train_idx) + len(indices) <= train_cutoff:
            train_idx.extend(indices)
        else:
            test_idx.extend(indices)
    return np.array(train_idx), np.array(test_idx)

## Cell 4 — per-target training loop

For each of the 4 targets:
1. Subset to that target's rows
2. Train + evaluate twice — once with random split, once with scaffold split
3. Record R², RMSE, MAE on the held-out test set

Random forest hyperparameters are sensible defaults — not tuned. The point is a baseline, not the best possible RF model.

In [ ]:
def evaluate(X, y, split_type, scaffolds=None, seed=42):
    if split_type == "random":
        X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=seed)
    elif split_type == "scaffold":
        tr_idx, te_idx = scaffold_split_indices(scaffolds, frac_train=0.8)
        X_tr, X_te = X[tr_idx], X[te_idx]
        y_tr, y_te = y[tr_idx], y[te_idx]
    else:
        raise ValueError(split_type)

    model = RandomForestRegressor(
        n_estimators=300,
        max_depth=None,
        n_jobs=-1,
        random_state=seed,
    )
    model.fit(X_tr, y_tr)
    preds = model.predict(X_te)

    return {
        "n_train": int(len(y_tr)),
        "n_test":  int(len(y_te)),
        "r2":      float(r2_score(y_te, preds)),
        "rmse":    float(np.sqrt(mean_squared_error(y_te, preds))),
        "mae":     float(mean_absolute_error(y_te, preds)),
    }

results = []
for target_id, target_name in TARGETS.items():
    mask = (clean["target_chembl_id"] == target_id).values
    X_t = fps[mask]
    y_t = clean.loc[mask, "pic50"].values
    sc_t = clean.loc[mask, "scaffold"].tolist()

    if len(y_t) < 50:
        print(f"Skipping {target_name} ({len(y_t)} rows — too few for meaningful evaluation)")
        continue

    for split_type in ["random", "scaffold"]:
        m = evaluate(X_t, y_t, split_type, scaffolds=sc_t)
        m["target"] = target_name
        m["split"]  = split_type
        results.append(m)
        print(
            f"{target_name:7s} {split_type:9s}: "
            f"n_train={m['n_train']:5d}  n_test={m['n_test']:5d}  "
            f"R\u00b2={m['r2']:6.3f}  RMSE={m['rmse']:.3f}  MAE={m['mae']:.3f}"
        )

results_df = pd.DataFrame(results)

## Cell 5 — side-by-side comparison

Random vs scaffold split, all targets in one table. The R² gap (random minus scaffold) is the most informative number — large gap means the model is leaning heavily on similarity to training data, small gap means it's actually learning structure-activity patterns that generalize.

Realistic expectations for this baseline:
- **Random split R²:** typically 0.5–0.8 for well-studied targets (EGFR, BRAF)
- **Scaffold split R²:** typically 0.2–0.5 — much harder, because the test set has novel chemistries
- KRAS might be lower across the board due to the historically weak compound distribution

In [ ]:
pivot = results_df.pivot(index="target", columns="split", values=["r2", "rmse", "mae"])
pivot

## Cell 6 — save metrics

Phase 3 will load `reports/baseline_rf_metrics.json` to print the GNN's metrics alongside these baselines, so we can see exactly how much (if any) the GNN improves things.

In [ ]:
with open(METRICS_PATH, "w") as f:
    json.dump(results, f, indent=2)

print(f"Saved {len(results)} metric records to {METRICS_PATH}")
print()
print("Contents:")
print(json.dumps(results, indent=2))